# Skip-gram untuk Klasifikasi Data Berita Menggunakan Naive Bayes

## 1. Import Library dan Membaca Dataset

Pada tahap ini dilakukan import library yang diperlukan untuk proses pengolahan teks, pembuatan word embedding menggunakan Word2Vec Skip-gram, klasifikasi menggunakan Naive Bayes, serta evaluasi model.

Dataset berita kemudian dibaca menggunakan pandas. Dataset terdiri dari 200 data dengan tiga kolom, yaitu id, isi_berita, dan label.

In [2]:
import pandas as pd
import numpy as np
import string
import re
import emoji

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from nltk.tokenize import word_tokenize

# Membaca dataset
df = pd.read_csv("dataset_berita.csv")
df

,id,isi_berita,label
0,1,Jepang - Austin Krajicek-Nikola Mektic berhasi...,sport
1,2,Jorge Martin sedang memperjuangkan titel juara...,sport
2,3,Rider Indonesia Veda Ega Pratama balapan di Ma...,sport
3,4,"Pelatih speed tim panjat tebing, Fitriyani, me...",sport
4,5,Pertamina Grand Prix of Indonesia 2026 sudah m...,sport
...,...,...,...
195,196,Otoritas Jasa Keuangan (OJK) menjatuhkan sanks...,finance
196,197,Kepala Badan Gizi Nasional (BGN) Sudaryono men...,finance
197,198,Utang pinjaman daring (pindar) masyarakat Indo...,finance
198,199,Jumlah investor pasar modal Indonesia terus be...,finance


## 2. Menyiapkan Stopword dan Stemming

Pada tahap ini digunakan library Sastrawi untuk melakukan preprocessing teks Bahasa Indonesia.

Stopword digunakan untuk menghapus kata-kata umum yang kurang memberikan informasi penting dalam proses klasifikasi. Sedangkan stemming digunakan untuk mengubah kata menjadi bentuk dasarnya.

Hasil dari tahap ini akan digunakan pada proses preprocessing berita.

In [3]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Membuat daftar stopword Bahasa Indonesia
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

# Membuat stemmer Bahasa Indonesia
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

## 3. Preprocessing Teks

Tahap preprocessing dilakukan untuk membersihkan dan mempersiapkan teks sebelum digunakan dalam proses Word2Vec.

Tahapan yang dilakukan meliputi:
1. Mengubah seluruh teks menjadi huruf kecil.
2. Menghapus tanda baca.
3. Menghapus angka.
4. Melakukan tokenisasi dengan memisahkan teks menjadi kata-kata.
5. Menghapus stopword.
6. Melakukan stemming untuk mengubah kata menjadi bentuk dasarnya.

Hasil akhir dari proses ini disimpan dalam token_preprocessing dan akan digunakan sebagai data masukan untuk proses Skip-gram.

In [4]:
token_preprocessing = []

for teks in df["isi_berita"].dropna():

    # 1. Mengubah teks menjadi huruf kecil
    teks = teks.lower()

    # 2. Menghapus tanda baca
    teks = teks.translate(str.maketrans('', '', string.punctuation))

    # 3. Menghapus angka
    teks = re.sub(r'\d+', '', teks)

    # 4. Tokenisasi
    kata = teks.split()

    # 5. Menghapus stopword
    kata = [k for k in kata if k not in stopwords]

    # 6. Stemming
    kata = [stemmer.stem(k) for k in kata]

    token_preprocessing.append(kata)

# Menampilkan hasil preprocessing 3 berita pertama
for i, token in enumerate(token_preprocessing[:3], 1):
    print(f"Berita {i}:")
    print(token)
    print()

Berita 1:
['jepang', 'austin', 'krajiceknikola', 'mektic', 'hasil', 'raih', 'gelar', 'juara', 'ganda', 'putra', 'japan', 'open', 'menang', 'raih', 'usai', 'tunduk', 'wakil', 'prancis', 'lewat', 'laga', 'sengit', 'foto', 'sport', 'krajicekmektic', 'sabet', 'gelar', 'ganda', 'putra', 'japan', 'open', 'selasa', 'okt', 'wib', 'suka', 'artikel', 'artikel', 'simpan']

Berita 2:
['jorge', 'martin', 'sedang', 'juang', 'titel', 'juara', 'dunia', 'dua', 'martin', 'punya', 'mimpi', 'jadi', 'juara', 'dunia', 'tiga', 'pabrikan', 'beda', 'mungkin', 'balap', 'spanyol', 'tang', 'gelar', 'pertama', 'sama', 'pramac', 'tunggang', 'motor', 'ducati', 'jorge', 'martin', 'kalah', 'rider', 'pabrikan', 'francesco', 'bagnaia', 'unggul', 'poin', 'klasemen', 'akhir', 'taji', 'jorge', 'martin', 'musim', 'motogp', 'kali', 'atas', 'aprilia', 'rsgp', 'martin', 'pimpin', 'klasemen', 'raih', 'poin', 'bayang', 'jago', 'ducati', 'marc', 'marquez', 'sejarah', 'balap', 'motor', 'kelas', 'premier', 'balap', 'mampu', 'tang',

## 4. Membuat Pasangan Target dan Konteks

Pada tahap ini dibuat pasangan kata target dan kata konteks sebagai bagian dari proses Skip-gram.

Parameter window = 1 digunakan sehingga setiap kata akan melihat satu kata di sebelah kiri dan satu kata di sebelah kanan sebagai konteks.

Hasilnya berupa tabel yang memiliki dua kolom, yaitu Target dan Konteks.

In [5]:
window = 1

data_target_context = []

for kalimat in token_preprocessing:

    for i, target in enumerate(kalimat):

        # Kata di sebelah kiri
        if i - window >= 0:
            data_target_context.append({
                "Target": target,
                "Konteks": kalimat[i - 1]
            })

        # Kata di sebelah kanan
        if i + window < len(kalimat):
            data_target_context.append({
                "Target": target,
                "Konteks": kalimat[i + 1]
            })

df_target_context = pd.DataFrame(data_target_context)

display(df_target_context)
print("Jumlah pasangan target-konteks:", len(df_target_context))

,Target,Konteks
0,jepang,austin
1,austin,jepang
2,austin,krajiceknikola
3,krajiceknikola,austin
4,krajiceknikola,mektic
...,...,...
94353,lindung,buat
94354,lindung,rakyat
94355,rakyat,lindung
94356,rakyat,halhns


Jumlah pasangan target-konteks: 94358


## 5. Membuat Model Word2Vec dengan Skip-gram

Pada tahap ini dibuat model Word2Vec menggunakan metode Skip-gram.

Parameter sg=1 menunjukkan bahwa Word2Vec menggunakan metode Skip-gram. Setiap kata akan direpresentasikan dalam bentuk vektor berdimensi 100 karena menggunakan vector_size=100.

Parameter window=2 digunakan untuk menentukan jumlah kata di sekitar kata target yang digunakan sebagai konteks. Kata yang muncul kurang dari 2 kali tidak dimasukkan ke dalam vocabulary karena menggunakan min_count=2.

Model dilatih selama 50 epoch.

In [6]:
model = Word2Vec(
    sentences=token_preprocessing,
    vector_size=100,
    window=2,
    min_count=2,
    sg=1,
    workers=1,
    epochs=50
)

print("Model Skip-gram berhasil dibuat")
print("Jumlah kata dalam vocabulary:", len(model.wv.index_to_key))

Model Skip-gram berhasil dibuat
Jumlah kata dalam vocabulary: 3307


In [7]:
model.save("word2vec_skipgram.model")

## 6. Menampilkan Word Embedding

Setelah model Skip-gram selesai dilatih, setiap kata dalam vocabulary memiliki representasi berupa vektor.

Karena vector_size=3, setiap kata memiliki tiga nilai vektor, yaitu Vektor 1, Vektor 2, dan Vektor 3.

Hasil word embedding kemudian ditampilkan dalam bentuk DataFrame agar lebih mudah dilihat dan dianalisis.

In [8]:
data_vektor = []

for kata in model.wv.index_to_key:
    vektor = model.wv[kata]

    data_vektor.append(
        [kata] + vektor.tolist()
    )

df_vektor = pd.DataFrame(
    data_vektor,
    columns=["Kata"] + [f"Vektor {i}" for i in range(1, 101)]
)

df_vektor

,Kata,Vektor 1,Vektor 2,Vektor 3,Vektor 4,Vektor 5,Vektor 6,Vektor 7,Vektor 8,Vektor 9,...,Vektor 91,Vektor 92,Vektor 93,Vektor 94,Vektor 95,Vektor 96,Vektor 97,Vektor 98,Vektor 99,Vektor 100
0,jadi,-0.321721,-0.495506,0.005316,0.154083,-0.215304,-0.359391,-0.000200,0.095453,-0.483400,...,0.411816,0.092040,0.108557,-0.255728,1.112196,-0.088175,-0.125070,-0.164197,0.080619,1.012470
1,indonesia,-0.610500,0.040997,-0.444014,0.543343,0.252091,0.152015,-0.610111,0.311128,-0.420920,...,-0.116455,0.070283,0.090875,-0.122408,-0.409209,0.015673,-0.093643,-0.324912,-0.635021,0.406816
2,sebut,-0.636993,0.523565,-0.425918,0.013591,0.162274,-0.151729,-0.073245,0.350768,-0.787422,...,-0.139038,0.565143,0.264134,-0.730172,0.801717,-0.025992,-0.026070,0.140442,0.795464,-0.531981
3,rp,-1.043547,0.654706,-0.201108,0.116747,-0.259023,-1.061798,-0.379345,1.691018,0.044331,...,0.016150,-0.348333,0.210792,-0.241467,1.481639,0.580530,0.847677,0.068470,0.747015,-0.520548
4,jakarta,-0.412345,-0.344053,-1.109796,0.397825,0.149076,0.624740,-0.101470,1.174752,0.106653,...,0.690720,-0.186534,0.065944,0.298157,-0.335846,-0.911801,0.977459,-0.111232,0.853932,-0.374321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3302,stoner,-0.099441,0.019502,-0.194018,-0.418644,0.210593,-0.197917,0.150410,0.456504,0.190453,...,0.249292,0.036416,-0.132104,0.242453,-0.055191,-0.155482,-0.258051,0.182623,-0.304329,0.180752
3303,duke,-0.134562,0.188502,0.067847,-0.176831,0.301693,-0.187733,0.107952,0.319540,0.228708,...,0.291012,0.074862,-0.062662,0.188101,-0.119242,-0.151231,-0.303357,0.348368,-0.306168,0.315235
3304,geoff,-0.213074,0.142670,0.155326,-0.181950,0.324638,-0.235681,0.069948,0.283074,0.234685,...,0.292643,0.100356,-0.038979,0.082172,-0.023628,-0.092465,-0.337139,0.293040,-0.231378,0.277332
3305,tunduk,-0.049472,-0.200916,-0.439609,-0.079894,-0.029516,-0.162341,-0.272426,0.080906,0.063743,...,0.056807,-0.060137,-0.336515,-0.254955,-0.002493,-0.108355,-0.349540,0.045202,-0.024895,0.069452


## 7. Menyimpan Hasil Word Embedding

Hasil word embedding disimpan dalam format CSV dengan nama hasil_skipgram.csv.

File ini berisi kata-kata yang terdapat dalam vocabulary beserta nilai vektornya.

In [9]:
df_vektor.to_csv(
    "hasil_skipgram.csv",
    index=False
)

print("Hasil word embedding berhasil disimpan.")

Hasil word embedding berhasil disimpan.


## 8. Membuat Vektor Setiap Berita

Word2Vec menghasilkan vektor untuk setiap kata, sedangkan proses klasifikasi membutuhkan representasi untuk setiap berita.

Oleh karena itu, dibuat fungsi buat_vektor_berita() untuk mengubah kumpulan vektor kata dalam satu berita menjadi satu vektor berita.

Vektor berita diperoleh dengan menghitung nilai rata-rata dari seluruh vektor kata yang terdapat dalam berita.

In [10]:
def buat_vektor_berita(tokens, model):

    vektor_kata = []

    for kata in tokens:

        if kata in model.wv:
            vektor_kata.append(model.wv[kata])

    if len(vektor_kata) > 0:
        return np.mean(vektor_kata, axis=0)

    else:
        return np.zeros(model.vector_size)

## 9. Membentuk Vektor untuk Seluruh Berita

Pada tahap ini fungsi buat_vektor_berita() diterapkan pada seluruh berita dalam dataset.

Hasilnya disimpan dalam variabel X. Setiap baris mewakili satu berita, sedangkan setiap kolom mewakili dimensi dari vektor berita.

Dengan 200 berita dan vector_size=3, ukuran data yang dihasilkan adalah 200 × 3.

In [11]:
vektor_berita = []

for tokens in token_preprocessing:

    vektor = buat_vektor_berita(tokens, model)

    vektor_berita.append(vektor)

X = np.array(vektor_berita)

print("Ukuran data:", X.shape)

Ukuran data: (200, 100)


In [12]:
def buat_vektor_berita(tokens, model):

    vektor_kata = []

    for kata in tokens:
        if kata in model.wv:
            vektor_kata.append(model.wv[kata])

    if len(vektor_kata) > 0:
        return np.mean(vektor_kata, axis=0)

    else:
        return np.zeros(model.vector_size)

In [13]:
kolom_vektor = [
    f"Vektor {i}"
    for i in range(1, model.vector_size + 1)
]

df_vsm = pd.DataFrame(
    X,
    columns=kolom_vektor
)

df_vsm.insert(
    0,
    "id",
    df["id"].values
)

df_vsm["label"] = df["label"].values

display(df_vsm)

,id,Vektor 1,Vektor 2,Vektor 3,Vektor 4,Vektor 5,Vektor 6,Vektor 7,Vektor 8,Vektor 9,...,Vektor 92,Vektor 93,Vektor 94,Vektor 95,Vektor 96,Vektor 97,Vektor 98,Vektor 99,Vektor 100,label
0,1,0.118057,-0.207871,-0.211587,0.179372,-0.036184,-0.248852,-0.413175,0.095686,0.088673,...,-0.048943,-0.141814,-0.057741,0.170496,-0.084158,-0.151851,0.088867,0.042703,0.231436,sport
1,2,-0.145816,-0.267674,-0.060074,-0.075820,-0.144901,-0.215788,-0.081281,0.274334,-0.180064,...,-0.015795,-0.265168,-0.058971,0.303861,0.202962,-0.390869,0.053398,-0.048245,0.173709,sport
2,3,-0.097634,-0.132962,-0.251747,0.048111,-0.084261,-0.191775,-0.217896,0.252177,0.010825,...,0.028998,-0.259248,-0.033430,0.241145,0.060028,-0.161840,0.014921,-0.066134,0.129118,sport
3,4,-0.170762,-0.042089,-0.029145,0.160867,-0.120597,-0.285619,-0.285229,0.168948,-0.308911,...,-0.112626,-0.285452,-0.101703,0.332969,0.206307,-0.120306,0.120604,0.159882,0.095685,sport
4,5,-0.094652,-0.131901,-0.180738,-0.091261,-0.287079,-0.051346,0.001293,0.229393,-0.106146,...,0.096999,-0.301318,0.166585,0.237099,0.160558,-0.586087,0.185464,-0.091586,0.055896,sport
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,-0.160536,0.381440,-0.168062,-0.042283,-0.014290,-0.339572,0.103667,0.506388,-0.177435,...,-0.078550,-0.016682,0.060470,0.491271,0.279015,0.085363,-0.107560,0.437987,-0.086160,finance
196,197,-0.065923,0.044109,0.028859,0.093955,0.230081,-0.465478,0.034411,0.529269,-0.399567,...,0.014476,-0.004727,0.234240,0.582028,0.273730,0.157620,-0.055085,0.324436,-0.035347,finance
197,198,-0.266268,0.290335,-0.040651,-0.081998,0.030053,-0.282371,0.102569,0.643008,-0.116871,...,0.049918,0.073515,0.031362,0.551171,0.389566,-0.001021,-0.128885,0.462793,-0.289151,finance
198,199,-0.210523,0.486026,-0.073050,0.062198,-0.050566,-0.266277,0.078887,0.586431,-0.063704,...,-0.124186,0.027323,0.060592,0.583695,0.387982,0.194273,-0.090332,0.456119,-0.148641,finance


## 10. Menentukan Fitur dan Label

Pada tahap ini data dipisahkan menjadi fitur dan label.

Variabel X berisi tiga nilai vektor dari setiap berita yang digunakan sebagai fitur klasifikasi.

Variabel y berisi label berita yang menjadi target klasifikasi.

In [14]:
# Fitur berupa vektor berita
X = df_vsm[kolom_vektor].values

# Target berupa label berita
y = df_vsm["label"].values

print("Ukuran fitur:", X.shape)
print("Jumlah label:", len(y))

Ukuran fitur: (200, 100)
Jumlah label: 200


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data training :", X_train.shape)
print("Data testing  :", X_test.shape)

Data training : (160, 100)
Data testing  : (40, 100)


## 11. Klasifikasi Menggunakan Naive Bayes

Pada tahap ini digunakan algoritma Gaussian Naive Bayes untuk melakukan klasifikasi berita.

Model dilatih menggunakan data training yang terdiri dari vektor berita dan labelnya. Setelah proses pelatihan selesai, model digunakan untuk memprediksi label pada data testing.

Hasil prediksi disimpan dalam variabel y_pred.

In [16]:
nb = GaussianNB()

# Melatih model
nb.fit(X_train, y_train)

# Melakukan prediksi
y_pred = nb.predict(X_test)

print("Prediksi selesai.")

Prediksi selesai.


In [17]:
import joblib

joblib.dump(nb, "naive_bayes.pkl")

['naive_bayes.pkl']

## 12. Evaluasi Menggunakan Accuracy dan Classification Report

Pada tahap ini dilakukan evaluasi terhadap hasil klasifikasi.

Akurasi digunakan untuk mengetahui persentase prediksi yang benar dari seluruh data testing.

Selain akurasi, digunakan classification report yang menampilkan nilai precision, recall, dan F1-score untuk setiap kelas.

Berdasarkan hasil pengujian, model memperoleh akurasi sebesar 95% pada 40 data testing.

In [18]:
akurasi = accuracy_score(y_test, y_pred)

print("Akurasi:", akurasi)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Akurasi: 0.975

Classification Report:
              precision    recall  f1-score   support

     finance       0.95      1.00      0.98        20
       sport       1.00      0.95      0.97        20

    accuracy                           0.97        40
   macro avg       0.98      0.97      0.97        40
weighted avg       0.98      0.97      0.97        40



## 13. Confusion Matrix

Confusion matrix digunakan untuk melihat jumlah prediksi benar dan salah pada masing-masing kelas.

Berdasarkan hasil confusion matrix, terdapat 20 data finance yang diprediksi dengan benar sebagai finance dan 18 data sport yang diprediksi dengan benar sebagai sport.

Terdapat 2 data sport yang diprediksi sebagai finance, sehingga jumlah prediksi yang benar adalah 38 dari 40 data testing.

In [19]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[20  0]
 [ 1 19]]


In [20]:
df_vsm.to_csv(
    "hasil_vector_space_model_skipgram.csv",
    index=False
)

print("Vector Space Model berhasil disimpan.")

Vector Space Model berhasil disimpan.
